# ✅ SOLUTION NOTEBOOK 2: Debugging a Broken Agent with Structured Logs

---

> ⚠️ **This is the reference solution.** Attempt the practice notebook first before looking here!

---


In [ ]:
!pip install gradio

## ✅ Step 1 Solution — Logger Implementation

**Key points:**
- Always use `"a"` (append) mode so you don't overwrite previous log entries
- `f.flush()` forces the write buffer to disk immediately — critical for debugging crashes
- The `error` flag changes the prefix from `INFO` to `ERROR` for easy grep-style filtering

In [ ]:
from datetime import datetime
import time
import random

LOG_FILE = "unit_converter_debug.log"

def log_to_file(message: str, error: bool = False):
    """
    Appends a timestamped log entry to the log file immediately.
    
    Format: YYYY-MM-DD HH:MM:SS | INFO/ERROR | message
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "ERROR" if error else "INFO"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"{timestamp} | {prefix} | {message}\n")
        f.flush()  # Force immediate write — critical for debugging crashes

# Verify it works
log_to_file("Logger initialized — solution notebook")
log_to_file("This is a test error", error=True)

with open(LOG_FILE, "r") as f:
    print(f.read())

---
## 🐛 Bug Analysis — What the Log Would Show

Before presenting the fixed code, here is what the log reveals for each bug:

### Bug #1 — Wrong Temperature Formula
```
INFO  | Received query: temp 100 C to F
INFO  | Routing to temperature converter: 100.0 F
INFO  | convert_temperature called: 100.0 F
INFO  | Query completed in 0.0001s
```
The agent **returns** `100°C = 148.00°F` instead of `212°F`.
No error is thrown — the wrong formula silently produces a wrong answer.
This is a **logic bug**, not a crash. The log shows the function was called and completed — but the answer is wrong.

### Bug #2 — Wrong km Conversion Factor
```
INFO  | Received query: km 5
INFO  | Routing to km converter: 5.0
INFO  | convert_km called: 5.0 km
INFO  | Query completed in 0.0001s
```
Returns `5 km = 8.0467 miles` — the km-per-mile factor was used instead of miles-per-km.
Again a **silent logic bug** — no crash, but a wrong answer.

### Bug #3 — Index Crash on Short Queries
```
INFO  | Received query: temp 32 C
ERROR | Error occurred: list index out of range
INFO  | Query completed in 0.0001s
```
The query `temp 32 C` tokenises to `['temp', '32', 'C']` (3 tokens, indices 0-2).
`tokens[4]` raises `IndexError: list index out of range`.
This is a **crash bug** — the ERROR log makes it immediately visible.

---

## ✅ Step 5 Solution — Fixed Agent

All three fixes are clearly marked with comments.

In [ ]:
# =========================================================
# FIXED UNIT CONVERTER AGENT
# =========================================================

def convert_temperature(value: float, from_unit: str) -> str:
    """Convert temperature between Celsius and Fahrenheit."""
    log_to_file(f"convert_temperature called: {value} {from_unit}")
    
    if from_unit.upper() == "C":
        # ✅ FIX #1: Changed - 32 to + 32 (correct C → F formula)
        result = (value * 9/5) + 32
        return f"{value}°C = {result:.2f}°F"
    elif from_unit.upper() == "F":
        result = (value - 32) * 5/9
        return f"{value}°F = {result:.2f}°C"
    else:
        raise ValueError(f"Unknown temperature unit: {from_unit}")


def convert_km(km: float) -> str:
    """Convert kilometres to miles."""
    log_to_file(f"convert_km called: {km} km")
    
    # ✅ FIX #2: Changed 1.60934 (km/mile) to 0.621371 (miles/km)
    miles = km * 0.621371
    return f"{km} km = {miles:.4f} miles"


def convert_kg(kg: float) -> str:
    """Convert kilograms to pounds."""
    log_to_file(f"convert_kg called: {kg} kg")
    pounds = kg * 2.20462
    return f"{kg} kg = {pounds:.4f} lbs"


def unit_converter_agent(query: str) -> str:
    """Routes a query to the correct conversion tool (FIXED version)."""
    log_to_file(f"Received query: {query}")
    start = time.time()

    try:
        tokens = query.strip().split()
        
        if "temp" in query.lower():
            value = float(tokens[1])
            # ✅ FIX #3: Changed tokens[4] → tokens[2]
            # 'temp 100 C'     → tokens = ['temp', '100', 'C']  → index 2 = 'C'  ✓
            # 'temp 100 C to F' → tokens = ['temp', '100', 'C', 'to', 'F'] → index 2 = 'C' ✓
            from_unit = tokens[2]
            log_to_file(f"Routing to temperature converter: {value} {from_unit}")
            answer = convert_temperature(value, from_unit)

        elif "km" in query.lower():
            value = float(tokens[1])
            log_to_file(f"Routing to km converter: {value}")
            answer = convert_km(value)

        elif "kg" in query.lower():
            value = float(tokens[1])
            log_to_file(f"Routing to kg converter: {value}")
            answer = convert_kg(value)

        else:
            answer = f"I don't understand '{query}'. Try: temp 100 C, km 5, kg 70"
            log_to_file("No matching tool found")

    except Exception as e:
        log_to_file(f"Error occurred: {e}", error=True)
        answer = f"Agent failed: {e}"

    finally:
        elapsed = time.time() - start
        log_to_file(f"Query completed in {elapsed:.4f}s")

    return answer


# Quick unit tests before launching Gradio
print("TESTS:")
print(unit_converter_agent("temp 100 C to F"))  # Should be 212.00°F
print(unit_converter_agent("temp 32 C"))          # Should be 89.60°F, no crash
print(unit_converter_agent("km 5"))               # Should be 3.1069 miles
print(unit_converter_agent("kg 70"))              # Should be 154.3234 lbs
print(unit_converter_agent("hello"))              # Unknown query message

## ✅ Step 6 Solution — Fixed Gradio Interface

In [ ]:
import gradio as gr

log_to_file("=" * 50)
log_to_file("Starting FIXED Unit Converter Agent")

demo_fixed = gr.Interface(
    fn=unit_converter_agent,
    inputs=gr.Textbox(lines=2, placeholder="Try: temp 100 C  |  km 5  |  kg 70"),
    outputs="text",
    title="Unit Converter Agent (FIXED)",
    description="All three bugs fixed. Check unit_converter_debug.log for clean INFO-only entries."
)

demo_fixed.launch()

---
## 📝 Sample Answers for Post-Fix Reflection

**Q1 — Are there any ERROR entries in the new log?**  
No. After applying the three fixes, all five test queries complete without any `ERROR` prefix entries. The log should show only `INFO` lines with successful tool calls and completion times.

**Q2 — How did the log file help you locate bugs faster?**  
The log file let us pinpoint the exact function that was called and the exact error message — without re-reading all the code. Bug #3 (the IndexError) was instantly identifiable from the `ERROR` line. Bugs #1 and #2 required comparing logged outputs against expected values, which was possible because the agent logged both the input value and routing decision.

**Q3 — Additional logging improvements:**  
- Log the actual result before returning (e.g., `log_to_file(f"Result: {answer}")`) so the log contains the full input-output pair
- Log the token list so parsing issues are immediately visible
- Add a log entry at startup with the agent version number
- Use log levels (DEBUG / INFO / WARNING / ERROR / CRITICAL) with Python's built-in `logging` module for more powerful filtering

---
## ✅ Bonus Solution — miles_to_km Tool

In [ ]:
def convert_miles(miles: float) -> str:
    """Convert miles to kilometres."""
    log_to_file(f"convert_miles called: {miles} miles")
    km = miles * 1.60934
    return f"{miles} miles = {km:.4f} km"


def unit_converter_agent_v2(query: str) -> str:
    """Extended agent — includes miles conversion."""
    log_to_file(f"Received query: {query}")
    start = time.time()

    try:
        tokens = query.strip().split()
        
        if "temp" in query.lower():
            value = float(tokens[1])
            from_unit = tokens[2]
            answer = convert_temperature(value, from_unit)
        elif "km" in query.lower():
            value = float(tokens[1])
            answer = convert_km(value)
        elif "kg" in query.lower():
            value = float(tokens[1])
            answer = convert_kg(value)
        elif "miles" in query.lower():
            # Validate that the second token is actually a number
            if len(tokens) < 2 or not tokens[1].replace('.', '', 1).isdigit():
                raise ValueError(f"Expected a number after 'miles', got: {tokens[1] if len(tokens) > 1 else 'nothing'}")
            value = float(tokens[1])
            log_to_file(f"Routing to miles converter: {value}")
            answer = convert_miles(value)
        else:
            answer = f"I don't understand '{query}'. Try: temp 100 C, km 5, kg 70, miles 10"
            log_to_file("No matching tool found")

    except Exception as e:
        log_to_file(f"Error occurred: {e}", error=True)
        answer = f"Agent failed: {e}"

    finally:
        elapsed = time.time() - start
        log_to_file(f"Query completed in {elapsed:.4f}s")

    return answer


# Test both valid and invalid miles input
print(unit_converter_agent_v2("miles 10"))    # 10 miles = 16.0934 km
print(unit_converter_agent_v2("miles abc"))   # ERROR log + friendly message

# Print relevant log section
with open(LOG_FILE, "r") as f:
    lines = f.readlines()
    print("\n--- Last 10 log entries ---")
    print("".join(lines[-10:]))